# 02 · Automatic prefix caching with chained block hashes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/vllm/02_prefix_cache.ipynb) &nbsp; 📖 Concept deck: [Automatic prefix caching](https://nilesh-salpe.github.io/visual-ai/vllm/05-prefix-caching/) &nbsp; ⏱️ Suggested time: **35 min**

**You will build:** hashing each full KV block together with its parent block's hash, looking up the longest cached prefix of a new prompt, and measuring how much prefill a shared system prompt saves.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


import hashlib, pickle, math
BLOCK = 16
def H(obj):
    """vLLM's default prefix_caching_hash_algo is sha256 over a pickled tuple."""
    return hashlib.sha256(pickle.dumps(obj)).digest()
NONE_HASH = b"\x00" * 32
SYSTEM = list(range(1000, 1000 + 70))           # a 70-token shared system prompt

## Task 1: Chained block hashes

For a token list, return the hashes of its **full** blocks only (a trailing partial block is not cached). Block i's hash is `H((parent_hash, tuple(block_tokens), None))` with `parent_hash = NONE_HASH` for the first block, as in `hash_block_tokens` in vLLM.
Chaining means equal blocks at different positions, or after different prefixes, get different hashes.

In [ ]:
def block_hashes(tokens, block=BLOCK):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_block_hashes(f):
    a = f(list(range(40)))
    assert len(a) == 2, "40 tokens = 2 full blocks + 8 uncached tokens"
    b = f(list(range(32)) + [7] * 5)
    assert a == b, "same first 32 tokens → same two hashes"
    c = f([99] + list(range(1, 40)))
    assert c[0] != a[0] and c[1] != a[1], "a different first block changes every later hash (chaining)"

check("block_hashes", _test_block_hashes, block_hashes)

## Task 2: Look up the longest cached prefix

`PrefixCache` maps block hash → block id. `lookup(tokens)` returns the number of prompt tokens whose KV can be reused: walk the prompt's block hashes from the start and stop at the first miss. `insert(tokens)` registers all full blocks of a finished prefill (assign new ids 0, 1, 2, …).
vLLM never reuses the very last token's KV for a full hit: the model must compute at least one token to produce logits. Cap the reusable count at `len(tokens) - 1`.

In [ ]:
class PrefixCache:
    def __init__(self):
        self.map, self.next_id = {}, 0
    def insert(self, tokens):
        # TODO
        raise NotImplementedError
    def lookup(self, tokens):
        # TODO
        raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_PrefixCache(cls):
    hb = globals().get("block_hashes_solution") or block_hashes
    pc = cls(); pc.insert(SYSTEM + [1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
    assert pc.lookup(SYSTEM + [50, 51, 52]) == 64, "70-token system prompt → 4 full blocks (64 tokens) reusable"
    assert pc.lookup([5] + SYSTEM) == 0, "a different first token invalidates everything"
    assert pc.lookup(SYSTEM[:32]) == 31, "exact full hit: keep one token to compute"

check("PrefixCache", _test_PrefixCache, PrefixCache)

## Task 3: How much prefill does caching save?

Simulate `n_users` chat requests, each = `SYSTEM` + a unique 30-token user message (use token ids 5000 + 30·u + j). Process them in order with a shared `PrefixCache`, inserting each after its prefill. Return the fraction of prompt tokens that did **not** need prefill.

In [ ]:
def prefill_saved(n_users=20):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_prefill_saved(f):
    s = f(20)
    total = 20 * 100; expected = 19 * 64 / total
    assert abs(s - expected) < 1e-9, f"19 of 20 requests reuse 64 cached tokens: {expected:.3f}, got {s:.3f}"

check("prefill_saved", _test_prefill_saved, prefill_saved)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Chained block hashes
def block_hashes_solution(tokens, block=BLOCK):
    out, parent = [], NONE_HASH
    for i in range(len(tokens) // block):
        parent = H((parent, tuple(tokens[i * block:(i + 1) * block]), None)); out.append(parent)
    return out

check("block_hashes (reference)", _test_block_hashes, block_hashes_solution)

In [ ]:
# @title Solution for Task 2: Look up the longest cached prefix
class PrefixCache_solution:
    def __init__(self):
        self.map, self.next_id = {}, 0
    def insert(self, tokens):
        for h in block_hashes_solution(tokens):
            if h not in self.map:
                self.map[h] = self.next_id; self.next_id += 1
    def lookup(self, tokens):
        n = 0
        for h in block_hashes_solution(tokens):
            if h not in self.map: break
            n += BLOCK
        return min(n, len(tokens) - 1)

check("PrefixCache (reference)", _test_PrefixCache, PrefixCache_solution)

In [ ]:
# @title Solution for Task 3: How much prefill does caching save?
def prefill_saved_solution(n_users=20):
    pc, saved, total = PrefixCache_solution(), 0, 0
    for u in range(n_users):
        prompt = SYSTEM + [5000 + 30 * u + j for j in range(30)]
        saved += pc.lookup(prompt); total += len(prompt); pc.insert(prompt)
    return saved / total

check("prefill_saved (reference)", _test_prefill_saved, prefill_saved_solution)

---
## Experiments
**System prompt length vs savings.** Only full blocks are cached, so a system prompt just over a block boundary wastes the remainder. Longer shared prefixes (few-shot examples, documents in RAG, multi-turn chats) save the most.

In [ ]:
for sys_len in [10, 16, 70, 500, 2000]:
    SYSTEM = list(range(1000, 1000 + sys_len))
    print(f"shared prefix {sys_len:5d} tokens → {prefill_saved_solution(50):.1%} of prompt tokens skip prefill")

---
## Takeaways
* vLLM hashes each full block with its parent's hash (sha256 by default), so a block hash identifies the entire prefix up to it.
* A new request reuses the longest run of matching block hashes; partial blocks and the last token are always recomputed.
* Prefix caching is on by default in vLLM V1 (`enable_prefix_caching=True`) and costs almost nothing when there are no hits.